# Chapter 04 — Machine Learning Advanced

Di chapter 03 kamu sudah melatih dan mengevaluasi model dengan scikit-learn. Sekarang kita naikkan levelnya: di chapter ini kita akan membahas **teknik lanjutan** yang membedakan praktisi ML yang sudah berpengalaman dari yang baru lulus tutorial.

Bayangkan chapter 03 adalah mengendarai mobil di jalan tol — fitur巡航控制, jalan lurus, tidak banyak hal yang perlu dipikirkan. Chapter 04 ini seperti mengendarai mobil yang sama tapi sekarang di **jalur pegunungan**: ada tanjakan, tikungan tajam, kabut tebal, dan jalan licin. Kamu tetap pakai mobil yang sama, tapi sekarang kamu harus tahu kapan harus ganti gigi, kapan harus rem, kapan harus buka lampu kabut. Teknik-teknik di chapter 04 adalah "skill ekstra" untuk menghadapi medan yang lebih sulit.

Sepanjang chapter ini kita akan main di empat front secara bergantian: **fitur** (bagaimana membuat dan menyaring fitur), **hyperparameter** (bagaimana mencari setting terbaik), **ensemble** (bagaimana menggabungkan banyak model), dan **kasus khusus data** (time series dan data tidak seimbang). Di akhir, semua disatukan dalam satu mini project end-to-end.

Prasyarat: pastikan kamu sudah nyaman dengan materi chapter 01 (Python dasar), chapter 02 (NumPy, Pandas, matplotlib), dan chapter 03 (scikit-learn, train/test split, pipeline, Logistic Regression, KNN, Decision Tree, Random Forest, Linear/Ridge/Lasso Regression). Kalau ada yang masih goyah, ada baiknya buka lagi notebook chapter 03 sebentar — chapter 04 akan sangat sering merujuk ke pola-pola yang sudah kita tetapkan di sana.

## Library yang Dipakai di Chapter Ini

Di chapter 03 kita sudah pakai `numpy`, `pandas`, `matplotlib`, dan `scikit-learn`. Sekarang kita tambah empat pustaka baru. **XGBoost** adalah implementasi gradient boosting yang sering jadi pemenang di kompetisi Kaggle. **LightGBM** dari Microsoft, secara umum lebih cepat dari XGBoost untuk dataset besar. **Optuna** adalah framework hyperparameter tuning dengan Bayesian optimization — lebih pintar dari GridSearch untuk ruang pencarian besar. **Imbalanced-learn** menyediakan SMOTE dan berbagai teknik resampling untuk dataset tidak seimbang.

Kalau kamu belum install semuanya, buka terminal dan jalankan: `pip install xgboost lightgbm optuna imbalanced-learn`. Setelah itu, mari kita setup environment dulu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

Cell di atas mengimpor library yang akan kita pakai di section 1 dan 2. Library tambahan (xgboost, lightgbm, optuna, imblearn) akan kita import belakangan saat section-nya tiba — supaya tidak menumpuk di awal dan membuat kamu overwhelmed.

Perhatikan juga `np.random.seed(42)` — ini menjamin "keacakan" kita reproducible. Coba hapus baris itu dan jalankan ulang cell bagian machine learning: kamu akan dapat angka yang berbeda-beda. Dengan seed yang sama, semua orang di dunia akan mendapat hasil yang persis sama untuk kode yang sama. Ini praktik standar di data science untuk reproducibility.

---

# Section 1 — Feature Engineering: Membuat Fitur Baru dari Data Mentah

Dataset mentah jarang sekali langsung cocok untuk model. Coba bayangkan kamu punya data karyawan dengan kolom `tahun_lahir` dan `tahun_masuk_kerja`. Sendirian, masing-masing kolom tidak banyak bicara. Tapi begitu kamu hitung `pengalaman = tahun_sekarang - tahun_masuk_kerja`, tiba-tiba model punya informasi yang lebih bermakna. **Feature engineering** adalah seni membuat fitur baru dari fitur yang sudah ada, sedemikian rupa sehingga model bisa menemukan pola dengan lebih mudah.

Ada tiga jurus utama yang akan kita pelajari di section ini. **Pertama**, fitur interaksi: mengalikan dua fitur untuk menangkap efek gabungan (misalnya `luas = panjang × lebar` di data properti, atau `pendapatan_per_anggota = pendapatan / jumlah_anggota_keluarga` di data kredit). **Kedua**, transformasi log: untuk data yang skewed berat (misalnya pendapatan, harga rumah, jumlah kunjungan web), log bisa "meratakan" distribusinya sehingga model linear tidak kewalahan. **Ketiga**, polynomial features: menambah pangkat dua atau pangkat tiga dari fitur untuk menangkap hubungan non-linear (misalnya usia squared untuk menangkap efek parabola — terlalu muda dan terlalu tua keduanya berisiko di dunia asuransi).

Kunci penting: feature engineering bukan sulap. Tidak ada formula ajaib yang berlaku universal. Yang ada adalah **pemahaman domain** — kamu perlu tahu data kamu bicara tentang apa. Tapi kabar baiknya, ada beberapa pola yang berulang di banyak kasus, dan kita akan latihan mengidentifikasi serta mengaplikasikan pola-pola itu.

Mari kita mulai dari fitur interaksi. Kita akan pakai dataset sintetis yang menggambarkan kasus kredit: ada `pendapatan` (rupiah per bulan) dan `jumlah_tanggungan` (orang). Coba ketik kode di cell bawah ini untuk membuat fitur baru `beban_per_orang`.

In [ ]:
df = pd.DataFrame({
    'pendapatan': [5_000_000, 8_000_000, 12_000_000, 3_500_000],
    'tanggungan': [2, 4, 1, 5]
})
df['beban_per_orang'] = df['pendapatan'] / (df['tanggungan'] + 1)
print(df)

Lihat outputnya: `beban_per_orang` adalah pendapatan dibagi `(tanggungan + 1)`. Kenapa `+ 1`? Karena kalau ada karyawan lajang dengan `tanggungan = 0`, kita tidak ingin terjadi division by zero. Trik `+ 1` (atau lebih umum `+ epsilon` dengan epsilon kecil) adalah pola standar untuk fitur yang bisa bernilai nol di penyebut.

Sekarang coba bayangkan kasus berbeda. Misalkan ada data `harga` dan `jumlah_pembelian`. Fitur `total_belanja = harga * jumlah_pembelian` akan lebih informatif daripada dua kolom terpisah, karena model sering kesulitan mengalikan dua fitur secara internal — model linear pada dasarnya menjumlahkan, bukan mengalikan. Dengan menyediakan hasil kalinya secara eksplisit, kita mempermudah pekerjaan model.

Untuk transformasi log, kita pakai `np.log1p` (log natural dari `x + 1`). Fungsi `log1p` lebih aman dari `np.log` biasa karena `log(0)` itu tak terdefinisi, sedangkan `log1p(0) = 0` dengan mulus. Coba ketik cell berikut untuk melihat efeknya pada data yang skewed.

In [ ]:
harga = np.array([100_000, 1_000_000, 100_000_000])
print('Sebelum log:', harga)
print('Setelah log1p:', np.log1p(harga).round(2))

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.hist(harga); plt.title('Skewed')
plt.subplot(1, 2, 2)
plt.hist(np.log1p(harga)); plt.title('Setelah log1p')
plt.tight_layout()

Amati histogram kiri: distribusinya miring ke kanan (right-skewed) — ada satu nilai 100 juta yang "menjauh" dari yang lain. Model linear sangat tidak nyaman dengan distribusi seperti ini karena error kuadrat jadi didominasi oleh outlier. Setelah log1p (histogram kanan), distribusinya jadi lebih simetris. Model linear bisa bekerja jauh lebih baik di skala log.

Untuk polynomial features, scikit-learn punya `PolynomialFeatures`. Mari kita coba — tapi sebelumnya, pikirkan: data kita kali ini akan non-linear, jadi model linear biasa akan gagal total. Kita butuh fitur tambahan untuk membantu model.

In [ ]:
X_linier = np.array([[1], [2], [3], [4], [5]])
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_linier)
print('Sebelum:', X_linier.ravel())
print('Sesudah:', X_poly)

Lihat outputnya: `PolynomialFeatures(degree=2)` mengambil satu kolom `x` dan mengubahnya jadi dua kolom: `x` dan `x²`. Sekarang kalau kita pakai regresi linear di `X_poly`, dia bisa menangkap hubungan parabola — karena pada dasarnya `y = a*x + b*x²` adalah persamaan parabola.

Parameter `include_bias=False` menghilangkan kolom yang isinya cuma 1 semua (konstanta). Konstan sudah ditangani oleh model itu sendiri (intercept), jadi tidak perlu dihitung dua kali. Coba ganti ke `True` dan jalankan ulang — kamu akan lihat kolom tambahan yang isinya 1.0 semua.

Satu catatan penting: polynomial degree yang lebih tinggi (3, 4, 5) membuat fitur eksponensial — `degree=10` dari 1 fitur jadi 11 fitur, dan `degree=10` dari 5 fitur jadi 1001 fitur. Komputasinya meledak, dan risiko overfitting juga meningkat. Aturan praktis: mulai dari `degree=2` atau `degree=3`, lihat hasilnya, baru naikkan kalau perlu.

Coba modifikasi cell polynomial di atas — ganti `X_linier` jadi `np.array([[10], [20], [30], [40], [50]])` dan lihat apa yang berubah. Apakah nilai polinomialnya juga ikut bergeser?

In [ ]:
X_linier = np.array([[10], [20], [30], [40], [50]])
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_linier)
print('Sebelum:', X_linier.ravel())
print('Sesudah:', X_poly)

Coba kamu amati: kolom pertama `X_poly` adalah `x` (skala 10-50), dan kolom kedua adalah `x²` (skala 100-2500). Polinomialnya jadi ikut bergeser — kalau model linear dilatih di data ini, dia akan menyesuaikan koefisiennya dengan skala yang baru.

**Mini-check refleksi**: Tanpa melihat catatan, coba jelaskan dengan bahasamu sendiri kapan kita butuh `log1p` dan kapan kita butuh `PolynomialFeatures(degree=2)`. Apa bedanya?

---

# Section 2 — Feature Selection: Memilih Fitur yang Benar-Benar Penting

Setelah membuat banyak fitur baru (dan ditambah fitur asli yang bisa puluhan atau ratusan), kita perlu bertanya: **apakah semua fitur ini benar-benar berguna?** Jawabannya sering kali tidak. Banyak fitur yang redundan, banyak yang hampir konstan, banyak yang tidak relevan dengan target. Memasukkan fitur yang tidak berguna tidak hanya menambah beban komputasi, tapi juga bisa **memperburuk akurasi** karena model jadi punya banyak distraksi.

Bayangkan kamu sedang belajar untuk ujian. Kamu punya 1000 halaman catatan. Apakah semua halaman itu penting? Tidak — beberapa halaman berisi informasi duplikat, beberapa berisi detail kecil yang jarang keluar di ujian, dan beberapa bahkan salah. Feature selection adalah proses memilah catatan itu: menyingkirkan yang redundan, membuang yang tidak relevan, menyisakan yang benar-benar membantu prediksi.

Scikit-learn menyediakan tiga metode utama. **VarianceThreshold** menghapus fitur yang variansinya nyaris nol — kalau sebuah fitur nilainya hampir selalu sama (misalnya 99% baris nilainya 0), fitur itu tidak punya daya diskriminasi dan bisa dibuang. **SelectKBest** memilih K fitur dengan skor statistik tertinggi relatif terhadap target — cocok untuk kasus di mana kita tahu persis berapa fitur yang kita mau. **RFE (Recursive Feature Elimination)** lebih mahal tapi lebih akurat: dia melatih model berulang kali, setiap iterasi membuang fitur yang paling tidak penting, sampai tersisa jumlah yang kita mau.

Mari kita coba VarianceThreshold dulu. Kita buat dataset sintetis yang punya satu fitur konstan dan satu fitur bervariasi, lalu lihat apa yang dibuang.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

X = np.array([
    [1, 0, 5],
    [2, 0, 8],
    [3, 0, 3],
    [4, 0, 7],
    [5, 0, 2]
])
sel = VarianceThreshold(threshold=0.01)
X_baru = sel.fit_transform(X)
print('Bentuk sebelum:', X.shape)
print('Bentuk sesudah:', X_baru.shape)
print('Sisa kolom:', sel.get_support())

Lihat outputnya: `get_support()` mengembalikan array boolean `[True, False, True]`. Kolom ke-2 (yang isinya 0 semua) terdeteksi punya variansi nol, jadi dibuang. Threshold 0.01 berarti "buang fitur yang variansinya di bawah 0.01".

Penting: VarianceThreshold hanya melihat fitur X, bukan hubungannya dengan y. Jadi dia hanya menyingkirkan fitur yang tidak bervariasi — dia tidak tahu fitur mana yang penting untuk prediksi. Untuk itu kita butuh SelectKBest atau RFE.

Sekarang coba SelectKBest dengan skor F-statistik (anova). Metode ini mengukur "seberapa kuat hubungan linear antara fitur X dan target y". Semakin tinggi skornya, semakin informatif fitur itu untuk membedakan kelas target.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X = np.random.randn(200, 5)
y = (X[:, 0] + X[:, 2] > 0).astype(int)  # hanya fitur 0 dan 2 yang relevan

sel = SelectKBest(score_func=f_classif, k=2)
X_baru = sel.fit_transform(X, y)
print('Skor tiap fitur:', sel.scores_.round(2))
print('Fitur terpilih:', sel.get_support())

Perhatikan `sel.scores_` — fitur 0 dan fitur 2 punya skor jauh lebih tinggi (di atas 50, jauh di atas fitur lain yang skornya di bawah 1). Itulah kenapa `k=2` memilih keduanya. Ini contoh yang cukup bersih karena kita tahu ground truth-nya: fitur 0 dan 2 memang yang relevan, fitur lain adalah noise.

Tapi SelectKBest punya keterbatasan: dia hanya melihat hubungan **linear** atau **univariat** (satu fitur pada satu waktu). Dia tidak tahu kalau fitur 1 dan fitur 3 sebenarnya **sama-sama** diperlukan untuk memprediksi y. Untuk kasus seperti itu, kita butuh RFE.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=2)
X_baru = rfe.fit_transform(X, y)
print('Ranking (1 = paling penting):', rfe.ranking_)
print('Fitur terpilih:', rfe.get_support())

RFE melatih model pada semua fitur, mengidentifikasi fitur paling tidak penting, membuangnya, lalu mengulangi. Akhirnya dia memberi **ranking** — fitur dengan ranking 1 adalah yang lolos. Biayanya: kalau kamu punya 100 fitur dan ingin menyisakan 10, RFE melatih model 91 kali. Untuk dataset kecil ini tidak masalah, tapi untuk dataset besar bisa mahal.

**Mini-check refleksi**: Kapan kamu akan pakai `VarianceThreshold` vs `SelectKBest` vs `RFE`? Pikirkan trade-off antara kecepatan, akurasi, dan kemampuan menangkap hubungan non-linear.

---

# Section 3 — Hyperparameter Tuning: Mencari Setting Terbaik

Setiap model ML punya "kenop" yang bisa diputar-putar — di Random Forest ada `n_estimators` (jumlah pohon), `max_depth` (kedalaman maksimum), `min_samples_split` (minimum sampel untuk split), dan seterusnya. Pengaturan kenop-kenop ini disebut **hyperparameter** (jangan bingung dengan parameter model — parameter dipelajari dari data, hyperparameter diatur sebelum training). Pertanyaan besarnya: **kombinasi hyperparameter mana yang memberikan akurasi terbaik?**

Kamu bisa coba satu per satu secara manual — ubah `n_estimators` jadi 100, train, catat akurasi; ubah jadi 200, train, catat; dan seterusnya. Tapi itu akan makan waktu lama dan kemungkinan besar kamu tidak akan menemukan kombinasi optimal. Hyperparameter tuning adalah proses otomatis untuk mencari kombinasi terbaik.

Ada tiga metode utama. **GridSearchCV** mencoba **semua** kombinasi dari grid yang kamu tentukan — exhaustive tapi lambat. Kalau kamu punya 4 hyperparameter dengan masing-masing 5 nilai, total kombinasinya 5⁴ = 625. **RandomizedSearchCV** hanya mengambil sampel acak dari grid — jauh lebih cepat, hasilnya cukup dekat dengan GridSearch untuk kebanyakan kasus. **Optuna** adalah yang paling pintar: dia menggunakan Bayesian optimization, belajar dari percobaan sebelumnya untuk memilih kombinasi berikutnya yang paling mungkin bagus — jauh lebih efisien untuk ruang pencarian besar.

Mari kita mulai dengan GridSearchCV. Kita akan tuning Random Forest pada dataset klasifikasi sederhana.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_.round(3))

Lihat `param_grid`: 2 nilai untuk `n_estimators` dan 3 nilai untuk `max_depth` = total 6 kombinasi. `cv=3` berarti setiap kombinasi diuji dengan 3-fold cross-validation, jadi total ada 6 × 3 = 18 kali training. `n_jobs=-1` artinya pakai semua core CPU secara paralel.

Setelah fit, `grid.best_params_` memberi kombinasi hyperparameter terbaik, dan `grid.best_score_` memberi skor cross-validation-nya. Model terbaik ini sudah terlatih ulang pada seluruh data training — kamu bisa langsung pakai `grid.predict(X_test)` untuk memprediksi.

Sekarang coba RandomizedSearchCV — kita pakai grid yang lebih besar untuk menunjukkan keuntungannya.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10)
}
rs = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=10, cv=3, random_state=42, n_jobs=-1)
rs.fit(X_train, y_train)
print('Best params:', rs.best_params_)
print('Best CV score:', rs.best_score_.round(3))

Bedanya dengan GridSearch: di sini `param_dist` berisi **distribusi** (`randint(50, 200)` artinya integer acak antara 50 dan 200), bukan list nilai tetap. `n_iter=10` artinya hanya 10 kombinasi acak yang akan dicoba — jauh lebih sedikit dari ratusan atau ribuan kombinasi yang akan dieksplorasi GridSearch pada grid setara.

RandomizedSearch cocok untuk **eksplorasi awal** — saat kamu belum tahu rentang hyperparameter mana yang terbaik. Setelah dapat "neighborhood" yang bagus, baru kamu bisa switch ke GridSearch di area kecil tersebut untuk fine-tuning.

Sekarang Optuna. Library ini punya konsep **study** dan **trial**: setiap trial mencoba satu kombinasi, dan Optuna belajar dari trial sebelumnya untuk memilih kombinasi berikutnya yang paling menjanjikan.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    model = RandomForestClassifier(**params, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, show_progress_bar=False)
print('Best params:', study.best_params)
print('Best score:', study.best_value.round(3))

Struktur Optuna: kita definisikan fungsi `objective` yang menerima `trial` dan mengembalikan skor yang ingin dimaksimalkan (atau diminimalkan, tinggal ganti `direction`). Di dalam fungsi, `trial.suggest_int(name, low, high)` meminta Optuna untuk memilihkan integer. Optuna akan mencoba 15 kombinasi (`n_trials=15`) dan berhenti di yang terbaik.

Untuk dataset ini mungkin Optuna tidak banyak berbeda dari Randomized — perbedaannya baru terasa di **ruang pencarian yang lebih besar** (ratusan hyperparameter) atau saat satu training butuh waktu lama. Di situ Optuna bisa 10-100x lebih cepat dari GridSearch.

**Mini-check refleksi**: Misalkan kamu punya 5 hyperparameter dan 10 nilai untuk masing-masing. Berapa kombinasi yang akan dieksplorasi GridSearch? Kalau kamu hanya punya waktu untuk 50 trial, metode mana yang kamu pilih — GridSearch, Randomized, atau Optuna?

---

# Section 4 — Ensemble Methods: Banyak Model Lebih Kuat dari Satu

Logika dasar ensemble sederhana: kalau satu model bisa salah, mengapa tidak bertanya ke banyak model dan mengambil suara mayoritas? Prinsip ini sudah dikenal dalam kehidupan sehari-hari — ketika ragu, kita sering minta pendapat beberapa teman, bukan hanya satu. Di ML, ide yang sama diterjemahkan secara matematis: gabungkan prediksi dari beberapa model, dan hasilnya biasanya lebih akurat dari model tunggal mana pun.

Ada tiga varian utama yang akan kita pelajari. **Voting Classifier** adalah yang paling sederhana: ambil banyak model, setiap model memberikan prediksi, prediksi akhir adalah yang paling banyak dipilih (mayoritas suara). Cocok saat model-modelnya **beragam** (misalnya Logistic Regression + KNN + Decision Tree, bukan tiga Logistic Regression yang sama). **Stacking** lebih canggih: sebuah "meta-model" belajar bagaimana menggabungkan prediksi base model — jadi kalau di kasus tertentu Logistic Regression lebih akurat dan di kasus lain KNN, meta-model akan tahu kapan harus percaya siapa. **Gradient Boosting** (XGBoost, LightGBM) adalah standar industri: dia melatih model secara berurutan, di mana setiap model berikutnya memperbaiki kesalahan model sebelumnya.

Mari kita mulai dengan Voting. Kita akan gabungkan Logistic Regression, KNN, dan Random Forest pada dataset yang sama.

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42))
    ],
    voting='hard'
)
voting.fit(X_train, y_train)
print('Akurasi voting:', voting.score(X_test, y_test).round(3))

Lihat `voting='hard'`: ini artinya prediksi akhir adalah modus dari prediksi base model (mayoritas suara). Alternatifnya `voting='soft'` artinya kita merata-ratakan probabilitas dari base model, lalu mengambil kelas dengan probabilitas rata-rata tertinggi. `soft` biasanya lebih akurat, tapi mengharuskan semua base model punya `predict_proba` (semua classifier scikit-learn punya secara default).

Penting: kekuatan ensemble datang dari **keberagaman** model. Kalau kamu menggabungkan tiga Logistic Regression yang identik, ensemble tidak lebih baik dari satu Logistic Regression. Logikanya: tiga orang yang berpikiran sama tidak lebih baik dari satu orang yang berpikiran sama. Idealnya, gabungkan model dengan bias yang berbeda — linear (Logistic Regression) + berbasis instance (KNN) + berbasis pohon (Decision Tree).

Sekarang Stacking — dia mengambil voting selangkah lebih jauh dengan meta-model.

In [ ]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3
)
stacking.fit(X_train, y_train)
print('Akurasi stacking:', stacking.score(X_test, y_test).round(3))

Struktur Stacking: base models (Logistic Regression, KNN) masing-masing memprediksi, lalu prediksi mereka dijadikan **fitur baru** untuk meta-model (di sini Logistic Regression). Meta-model belajar: "kalau LR bilang 0 dan KNN bilang 0, kemungkinan besar jawabannya 0; tapi kalau LR bilang 0 dan KNN bilang 1, lebih baik jawab 1".

Penting: parameter `cv=3` di sini bukan untuk evaluasi, tapi untuk **menghasilkan prediksi base model**. Tanpa cv, base model akan memprediksi data yang mereka lihat saat training — meta-model akan overfit. Dengan cv, prediksi base model adalah out-of-fold, yang tidak bias.

Sekarang XGBoost dan LightGBM — standar industri untuk data tabular. Mari kita bandingkan di dataset yang sama.

In [ ]:
import xgboost as xgb
import time

xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric='logloss')
start = time.time()
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start
print(f'XGBoost: akurasi={xgb_model.score(X_test, y_test):.3f}, waktu={xgb_time:.2f}s')

XGBoost adalah implementasi gradient boosting yang sangat dioptimasi. `eval_metric='logloss'` adalah metrik evaluasi internal selama training — logloss untuk klasifikasi biner. Coba hapus parameter itu dan jalankan ulang, kamu akan dapat warning.

Sekarang LightGBM di cell berikutnya — umumnya lebih cepat, terutama di dataset besar dengan banyak fitur kategori.

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(n_estimators=100, max_depth=3, random_state=42, verbose=-1)
start = time.time()
lgb_model.fit(X_train, y_train)
lgb_time = time.time() - start
print(f'LightGBM: akurasi={lgb_model.score(X_test, y_test):.3f}, waktu={lgb_time:.2f}s')
print(f'Perbandingan: LightGBM {xgb_time/lgb_time:.1f}x lebih cepat dari XGBoost pada dataset ini')

Di dataset kecil ini perbedaannya tidak terlalu terasa. Tapi di dataset dengan 100rb+ baris dan 100+ fitur, LightGBM bisa 5-10x lebih cepat dari XGBoost. Keduanya sering menghasilkan akurasi yang mirip, jadi pilihan praktis: mulai dengan LightGBM, switch ke XGBoost kalau kamu butuh fitur yang hanya ada di XGBoost (misalnya GPU training).

**Mini-check refleksi**: Misalkan kamu menggabungkan 5 model identik (5 Logistic Regression dengan hyperparameter sama) dalam Voting Classifier. Apakah kamu mengharapkan akurasi ensemble lebih tinggi dari satu model? Mengapa atau mengapa tidak?

---

# Section 5 — Time Series: Prediksi Data yang Punya Urutan Waktu

Semua kasus di chapter 03 mengasumsikan data **i.i.d. (independent and identically distributed)** — artinya setiap baris tidak bergantung dengan baris lain. Tapi bagaimana kalau data kamu adalah data **time series**: harga saham harian, jumlah pengunjung website per jam, penjualan bulanan, atau curah hujan harian? Dalam kasus ini, **urutan waktu sangat penting** dan baris hari ini bergantung dengan baris kemarin.

Ini bukan sekadar detail — ini mengubah total cara kita memproses data. **Pertama**, kalau kita pakai `train_test_split` random seperti biasa, kita akan membocorkan informasi masa depan ke data training. Model akan "melihat" data tahun 2024 saat training, lalu diuji di data tahun 2023. Itu curang dan memberikan kita keyakinan palsu. **Kedua**, kita perlu membuat fitur yang menangkap pola waktu: rata-rata bergerak (rolling mean), lag features (nilai 7 hari lalu), jam/hari/bulan sebagai fitur kategorikal. **Ketiga**, kita butuh validator silang khusus: `TimeSeriesSplit` yang memastikan lipatan training selalu di masa lalu dan lipatan test selalu di masa depan.

Mari kita mulai dengan membuat dataset time series sintetis: penjualan harian dengan tren naik dan noise.

Kita akan buat dataset time series sederhana, lalu resample ke mingguan untuk melihat agregasi.

In [ ]:
tanggal = pd.date_range('2024-01-01', periods=60, freq='D')
penjualan = 100 + np.arange(60) * 0.5 + np.random.randn(60) * 5
ts = pd.Series(penjualan, index=tanggal)
print('Lima hari pertama:\n', ts.head())
print('\nResample ke mingguan (mean):\n', ts.resample('W').mean().round(1))

Lihat `ts.resample('W').mean()`: `W` artinya weekly, jadi kita mengambil rata-rata per minggu. `pd.date_range` dengan `freq='D'` menghasilkan 60 tanggal harian. Setelah resample, kita punya sekitar 9 titik mingguan (60 hari / 7).

Resample punya dua arah. **Downsampling** (harian → mingguan) butuh agregasi (`mean`, `sum`, `max`). **Upsampling** (harian → per jam) butuh interpolasi atau forward fill. Untuk data bisnis, downsampling jauh lebih umum — kita ingin melihat ringkasan mingguan/bulanan dari data harian yang sangat granular.

Sekarang rolling window — ini cara menghitung rata-rata bergerak yang sangat umum di analisis time series.

In [ ]:
ts_rolling = ts.rolling(window=7).mean()
print('Rolling 7-hari (10 nilai pertama, NaN karena window belum cukup):\n', ts_rolling.head(10).round(1))

`rolling(window=7).mean()` menghitung rata-rata dari 7 hari terakhir untuk setiap titik. Enam nilai pertama adalah NaN karena window 7 belum terisi. Ini smooths out noise — sangat berguna untuk melihat tren jangka panjang di balik fluktuasi harian.

Parameter `window` adalah hyperparameter penting. Window 7 cocok untuk pola mingguan (puncak di akhir pekan), window 30 untuk pola bulanan, window 365 untuk pola tahunan. Aturan praktis: window harus sepanjang satu siklus yang ingin kamu capture.

Sekarang lag features — fitur yang merupakan nilai masa lalu. Ini yang paling penting untuk prediksi time series: model diajarkan "jika 7 hari lalu penjualannya X, kemungkinan besar hari ini sekitar Y".

In [ ]:
df_ts = pd.DataFrame({'penjualan': ts})
df_ts['lag_1'] = df_ts['penjualan'].shift(1)
df_ts['lag_7'] = df_ts['penjualan'].shift(7)
df_ts['rolling_7'] = df_ts['penjualan'].rolling(7).mean()
print(df_ts.head(10).round(1))

Perhatikan `.shift(1)`: ini menggeser kolom ke bawah, jadi baris ke-2 berisi nilai baris ke-1. Baris pertama menjadi NaN. `lag_7` menggeser 7 baris, jadi untuk memprediksi hari ke-8 kita bisa pakai nilai hari ke-1.

Inilah pola standar time series ML: original fitur (penjualan hari ini) + lag features + rolling stats + fitur waktu (jam, hari, bulan). Setelah punya DataFrame seperti ini, kamu bisa perlakukan sebagai supervised learning biasa.

Sekarang split yang benar untuk time series — TimeSeriesSplit. Bedanya dengan train_test_split biasa: di sini lipatan training selalu di masa lalu.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

X_dummy = np.arange(20).reshape(-1, 1)
tscv = TimeSeriesSplit(n_splits=3)
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_dummy)):
    print(f'Fold {fold+1}: train={train_idx}, test={test_idx}')

Lihat outputnya: di setiap fold, data training selalu di masa lalu dan data test selalu di masa depan. Tidak pernah tumpang tindih, tidak pernah acak. Fold 1: train 0-11, test 12-14. Fold 2: train 0-14, test 15-17. Fold 3: train 0-17, test 18-19. Ini meniru skenario real-world: kamu melatih dengan data historis, lalu menguji dengan data masa depan.

**Mini-check refleksi**: Misalkan kamu melatih model dengan `train_test_split(X, y, test_size=0.2)` secara random pada data time series, dan mendapat akurasi 95%. Apakah kamu percaya model ini akan akurat di production? Mengapa?

---

# Section 6 — Imbalanced Dataset: Ketika Satu Kelas Mendominasi

Bayangkan kamu melatih model untuk mendeteksi transaksi fraud. Dari 100.000 transaksi, hanya 100 yang fraud (0.1%). Model yang selalu memprediksi "bukan fraud" akan mendapat akurasi 99.9% — terdengar sangat bagus bukan? Tapi model itu **tidak berguna** karena dia tidak pernah mendeteksi satu pun kasus fraud yang sebenarnya kita incar.

Ini adalah jebakan terbesar di ML: **akurasi bukan metrik yang tepat untuk data tidak seimbang**. Ketika satu kelas punya 99% data, model bodoh yang selalu tebak kelas mayoritas akan mendapat akurasi 99% — tanpa belajar apa-apa.

Solusinya ada tiga lapis. **Pertama**, gunakan metrik yang tepat: precision, recall, F1, atau ROC-AUC/PR-AUC — bukan akurasi. **Kedua**, atur biaya kesalahan berbeda untuk kelas berbeda: `class_weight='balanced'` membuat model "lebih peduli" pada kelas minoritas. **Ketiga**, seimbangkan data itu sendiri: **SMOTE** (Synthetic Minority Over-sampling Technique) membuat sampel sintetis kelas minoritas dengan menginterpolasi antara sampel yang ada.

Mari kita buat dataset imbalanced dan lihat masalahnya.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, weights=[0.95, 0.05], random_state=42)
print('Distribusi kelas:', np.bincount(y))
print('Persentase kelas 1:', y.mean().round(3))

Lihat outputnya: dari 1000 sampel, hanya 5% yang kelas 1 (kelas minoritas). Sekarang kita lihat apa yang terjadi kalau kita latih model tanpa menangani ini.

In [ ]:
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model_default = LogisticRegression(max_iter=1000)
model_default.fit(X_train, y_train)
print('Tanpa penanganan:')
print(classification_report(y_test, model_default.predict(X_test), zero_division=0))

Lihat baris "1" di classification report: precision, recall, dan f1-score semuanya 0.00! Artinya model memprediksi **tidak ada** pun sampel positif dengan benar. Akurasi secara keseluruhan tetap 95% — menyesatkan sekali. Sekarang kita coba `class_weight='balanced'`.

In [ ]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)
print('Dengan class_weight=balanced:')
print(classification_report(y_test, model_balanced.predict(X_test), zero_division=0))

Lihat perbedaannya: sekarang recall kelas 1 bukan 0.00 lagi. Model mulai "memperhatikan" kelas minoritas. Biaya kesalahan untuk kelas 1 dinaikkan secara proporsional dengan kebalikannya, sehingga model "lebih takut" salah memprediksi kelas 1.

Tapi kelemahannya: `class_weight='balanced'` adalah hyperparameter yang harus kamu tentukan sendiri, dan tidak semua model menerimanya. Alternatif yang lebih fleksibel: **SMOTE** — membuat sampel sintetis dari kelas minoritas.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print('Sebelum SMOTE:', np.bincount(y_train))
print('Sesudah SMOTE:', np.bincount(y_res))

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_res, y_res)
print('\nDengan SMOTE:')
print(classification_report(y_test, model_smote.predict(X_test), zero_division=0))

Lihat outputnya: sebelum SMOTE ada 760 sampel kelas 0 dan 40 sampel kelas 1. Setelah SMOTE, kelas 1 "dibuat" menjadi 760 juga — seimbang. SMOTE bekerja dengan mengambil sampel kelas 1 yang ada, lalu membuat sampel baru di garis lurus antara sampel itu dan tetangga kelas 1 terdekatnya (interpolasi).

Penting: SMOTE hanya boleh diterapkan di **data training**, bukan data test. Kalau kamu terapkan di data test, kamu akan membuat data test yang tidak representatif dengan dunia nyata (di mana data test datang dari distribusi asli yang tidak seimbang).

**Mini-check refleksi**: Misalkan kamu deploy model fraud detection ke production. Apakah kamu lebih memilih akurasi 99% tapi recall fraud 0%, atau akurasi 95% tapi recall fraud 80%? Pikirkan trade-off-nya.

---

# Section 7 — ROC, Precision-Recall, dan Threshold Tuning

Setiap classifier biner sejatinya bekerja dengan **threshold**: model menghasilkan probabilitas (misalnya 0.73), lalu kalau probabilitas ≥ 0.5 kita prediksi kelas 1, kalau < 0.5 kita prediksi kelas 0. Tapi 0.5 itu **pilihan default**, bukan aturan alam. Kapan harus mengubah threshold ini? Jawabannya tergantung pada **biaya relatif** dari false positive vs false negative di kasus kamu.

Untuk memahami perilaku model di **semua** threshold yang mungkin, kita pakai dua kurva. **ROC curve** (Receiver Operating Characteristic) plot True Positive Rate (recall) vs False Positive Rate untuk setiap threshold. **AUC** (Area Under Curve) adalah ringkasan satu angka — semakin mendekati 1, semakin bagus. Cocok untuk data **balanced**.

Untuk data **imbalanced**, ROC kadang terlalu optimis karena FPR (False Positive Rate) bisa kecil secara proporsi meskipun jumlah false positive-nya banyak. Solusinya: **Precision-Recall curve**. PR-AUC lebih informatif untuk imbalanced karena langsung menunjukkan trade-off precision dan recall.

Mari kita hitung ROC-AUC dan PR-AUC pada dataset breast cancer (balanced) dan visualisasikan kedua kurva secara berdampingan.

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)
model_bc = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
proba = model_bc.predict_proba(X_te)[:, 1]

fpr, tpr, _ = roc_curve(y_te, proba)
precision, recall, _ = precision_recall_curve(y_te, proba)
print(f'ROC-AUC: {auc(fpr, tpr):.3f}')
print(f'PR-AUC:  {average_precision_score(y_te, proba):.3f}')

Pada dataset balanced, ROC-AUC dan PR-AUC biasanya mirip (keduanya mendekati 1 untuk model bagus). Sekarang mari kita visualisasikan ROC dan PR curve secara berdampingan.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.3f}')
ax[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('ROC Curve'); ax[0].legend()

ax[1].plot(recall, precision, label=f'AP = {average_precision_score(y_te, proba):.3f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall Curve'); ax[1].legend()
plt.tight_layout()

Garis putus-putus diagonal di ROC adalah "model acak" — kalau kurva model kamu ada di garis itu, model sama sekali tidak berguna. Model yang bagus punya kurva ROC yang naik cepat ke pojok kiri atas (TPR tinggi, FPR rendah).

Sekarang **threshold tuning** — ini teknik yang sangat praktis. Default threshold 0.5 sering bukan yang terbaik, terutama di data imbalanced. Mari kita cari threshold optimal untuk F1.

In [ ]:
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(y_te, (proba >= t).astype(int)) for t in thresholds]
best_t = thresholds[np.argmax(f1_scores)]
print(f'Threshold optimal untuk F1: {best_t:.2f}')
print(f'F1 di threshold default (0.5): {f1_score(y_te, (proba >= 0.5).astype(int)):.3f}')
print(f'F1 di threshold optimal:       {max(f1_scores):.3f}')

Lihat outputnya: threshold optimal bisa jadi 0.3, 0.4, atau 0.6 — tergantung data. Menggunakannya dapat memperbaiki F1 sampai beberapa poin persentase tanpa mengubah model sama sekali. Ini "free lunch" yang sering diabaikan.

**Mini-check refleksi**: Untuk kasus deteksi kanker (kita ingin **tidak melewatkan** kasus positif), apakah kita ingin threshold tinggi atau rendah? Untuk kasus email spam (kita ingin **tidak salah** menandai email penting sebagai spam), threshold-nya bagaimana?

---

# Section 8 — Mini Project End-to-End: Prediksi Churn Pelanggan

Sekarang kita satukan semua yang sudah dipelajari. Mini project ini akan melatih model untuk memprediksi churn (pelanggan yang akan berhenti berlangganan) pada dataset sintetis yang menggabungkan banyak tantangan: ada fitur numerik, fitur kategori, distribusi kelas tidak seimbang, dan banyak fitur yang bisa dipilih.

Pipeline lengkap yang akan kita bangun: (1) muat data, (2) pisahkan fitur dan target, (3) split train-test stratified, (4) preprocessing dengan ColumnTransformer (scaling untuk numerik, encoding untuk kategori), (5) feature engineering — buat fitur interaksi, (6) feature selection dengan SelectKBest, (7) train Random Forest dengan class_weight='balanced', (8) evaluasi dengan PR-AUC karena data imbalanced, (9) tuning threshold untuk F1, (10) simpan model dengan joblib untuk dipakai ulang di production.

Mari kita mulai dengan membuat dataset churn sintetis. Dataset ini akan punya 2000 baris pelanggan telco, dengan fitur numerik (usia, monthly_charges, tenure) dan fitur kategori (jenis_kontrak, metode_pembayaran). Distribusi churn sekitar 25%.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

np.random.seed(42)
n = 2000
df_churn = pd.DataFrame({
    'usia': np.random.randint(18, 70, n),
    'tenure_bulan': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 120, n).round(2),
    'total_charges': np.random.uniform(100, 8000, n).round(2),
    'jenis_kontrak': np.random.choice(['bulanan', '1_tahun', '2_tahun'], n),
    'metode_bayar': np.random.choice(['transfer', 'kartu_kredit', 'e_wallet'], n),
})
churn_prob = 0.25 + 0.3 * (df_churn['monthly_charges'] > 80).astype(int) - 0.2 * (df_churn['tenure_bulan'] > 24).astype(int)
df_churn['churn'] = (np.random.rand(n) < churn_prob).astype(int)
print('Bentuk data:', df_churn.shape)
print('Distribusi churn:\n', df_churn['churn'].value_counts())

Perhatikan `churn_prob`: kita sengaja membuat fitur `monthly_charges` tinggi dan `tenure_bulan` rendah meningkatkan probabilitas churn. Ini simulasi domain knowledge — di telco, pelanggan dengan tagihan bulanan tinggi yang baru sebentar berlangganan cenderung lebih mudah churn.

Distribusi churn sekitar 25-30% — ini **imbalanced**. Akurasi 75% saja terdengar biasa saja untuk data ini, tapi recall kelas 1 (churn) yang rendah akan jadi masalah besar untuk bisnis.

Sekarang feature engineering — kita buat fitur interaksi `charges_per_tenure`.

In [ ]:
df_churn['charges_per_tenure'] = df_churn['total_charges'] / (df_churn['tenure_bulan'] + 1)
print('Lima baris dengan fitur baru:')
print(df_churn[['total_charges', 'tenure_bulan', 'charges_per_tenure', 'churn']].head())

Sekarang kita bangun pipeline preprocessing. Kolom numerik akan di-scale, kolom kategori akan di-encode. ColumnTransformer memungkinkan kita melakukan keduanya dalam satu langkah.

In [ ]:
X = df_churn.drop('churn', axis=1)
y = df_churn['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['usia', 'tenure_bulan', 'monthly_charges', 'total_charges', 'charges_per_tenure']
cat_cols = ['jenis_kontrak', 'metode_bayar']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

Lihat `ColumnTransformer`: dia menerapkan `StandardScaler` ke kolom numerik dan `OneHotEncoder` ke kolom kategori, lalu menggabungkan hasilnya. `handle_unknown='ignore'` artinya kalau di data test ada kategori yang tidak ada di training, dia tidak error tapi diisi 0 semua.

Sekarang kita rangkai semua jadi satu pipeline akhir: preprocessor + feature selection + classifier.

In [ ]:
from sklearn.metrics import classification_report, average_precision_score

pipe = Pipeline([
    ('prep', preprocessor),
    ('select', SelectKBest(f_classif, k=5)),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

pipe.fit(X_train, y_train)
proba_test = pipe.predict_proba(X_test)[:, 1]
print('PR-AUC:', average_precision_score(y_test, proba_test).round(3))
print('\nLaporan klasifikasi (threshold default 0.5):')
print(classification_report(y_test, pipe.predict(X_test), zero_division=0))

Lihat `Pipeline`: tiga langkah berturut-turut — preprocess, select, classify. Ini memastikan semua transformasi diterapkan dengan benar saat prediksi (tidak ada data leakage). `SelectKBest(f_classif, k=5)` memilih 5 fitur terbaik dari 9 fitur (5 numerik + 4 one-hot dari 2 kategori).

Sekarang threshold tuning — karena data imbalanced, kita cari threshold optimal untuk F1.

In [ ]:
thresholds = np.arange(0.2, 0.8, 0.05)
f1s = [f1_score(y_test, (proba_test >= t).astype(int)) for t in thresholds]
best_t = thresholds[np.argmax(f1s)]
print(f'Threshold optimal: {best_t:.2f}')
print(f'F1 di threshold default: {f1_score(y_test, (proba_test >= 0.5).astype(int)):.3f}')
print(f'F1 di threshold optimal: {max(f1s):.3f}')

Coba kamu amati: threshold optimal sering bukan 0.5, terutama di data imbalanced. Ini berlaku untuk hampir semua kasus bisnis di mana biaya false positive dan false negative tidak simetris.

Terakhir — menyimpan model dengan `joblib` supaya bisa di-load ulang di production.

In [ ]:
import joblib

joblib.dump(pipe, 'model_churn.joblib')
loaded = joblib.load('model_churn.joblib')
print('Prediksi dari model yang di-load ulang:', loaded.predict(X_test.head(3)).tolist())

Setelah di-save, kamu bisa panggil `joblib.load('model_churn.joblib')` di script lain atau di service production, dan langsung dapat pipeline yang sudah terlatih lengkap dengan preprocessor dan classifier-nya. Ini cara standar deploy model ML ke production.

**Mini-check refleksi akhir chapter**: Coba kamu jelaskan dengan bahasamu sendiri mengapa kita pakai `class_weight='balanced'` di Random Forest, `SelectKBest` di tengah pipeline, dan `PR-AUC` untuk evaluasi — bukan akurasi. Kalau kamu bisa menjawab ketiganya, kamu sudah memahami alur berpikir data scientist di kasus imbalanced.

---

# Ringkasan Chapter 04

Selamat! Kamu sudah menyelesaikan chapter 04 — Machine Learning Advanced. Mari kita rangkum apa yang sudah dipelajari.

Di **Section 1** kita membahas feature engineering: membuat fitur baru dengan interaksi (perkalian/pembagian), transformasi log untuk data skewed, dan polynomial features untuk menangkap hubungan non-linear. Feature engineering adalah seni — tidak ada formula ajaib, tapi ada pola berulang yang bisa kamu kenali.

Di **Section 2** kita membahas feature selection: VarianceThreshold untuk membuang fitur yang hampir konstan, SelectKBest untuk memilih K fitur terbaik dengan skor statistik, dan RFE untuk eliminasi rekursif yang lebih akurat tapi lebih mahal.

Di **Section 3** kita membahas hyperparameter tuning: GridSearchCV (exhaustive), RandomizedSearchCV (sampel acak), dan Optuna (Bayesian optimization yang paling efisien). Ketiganya punya tempat masing-masing — pilih berdasarkan ukuran ruang pencarian dan budget komputasi.

Di **Section 4** kita membahas ensemble: Voting (mayoritas suara), Stacking (meta-model), dan gradient boosting (XGBoost, LightGBM — standar industri untuk data tabular). Kunci utama: ensemble bekerja kalau model-modelnya beragam.

Di **Section 5** kita membahas time series: resample untuk agregasi, rolling window untuk moving average, lag features untuk menangkap pola waktu, dan TimeSeriesSplit untuk validasi tanpa membocorkan masa depan.

Di **Section 6** kita membahas imbalanced dataset: akurasi bukan metrik yang tepat, class_weight='balanced' menaikkan biaya kelas minoritas, dan SMOTE membuat sampel sintetis untuk menyeimbangkan data.

Di **Section 7** kita membahas ROC-AUC, PR-AUC, dan threshold tuning: ROC untuk balanced, PR untuk imbalanced, dan threshold optimal bisa sangat berbeda dari 0.5 default.

Di **Section 8** kita menyatukan semuanya dalam mini project prediksi churn — pipeline lengkap dari data mentah sampai model yang siap di-deploy.

Setelah chapter ini, kamu siap untuk chapter 05: Deep Learning. Di sana kita akan membahas neural network, CNN untuk gambar, RNN/LSTM untuk sequence, dan transfer learning — semuanya dibangun di atas fondasi ML klasik yang sudah kamu kuasai di chapter 03 dan 04.